In [18]:
import numpy as np

In [19]:
class SkipgramModel:
    def __init__(self, vocab_size, embed_size, learning_rate = 0.1):
        self.v_size = vocab_size
        self.e_size = embed_size
        self.lr = learning_rate

        self.neg_table_size = 1000000
        self.neg_table = self._create_neg_table()
        self.neg_ptr = 0

        self.W1 = np.random.randn(self.v_size, self.e_size) * 0.1
        self.W2 = np.random.randn(self.v_size, self.e_size) * 0.1

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -15, 15)))

    def _create_neg_table(self):
        return np.random.randint(0, self.v_size, self.neg_table_size)

    def _get_neg_samples(self, count):
        if self.neg_ptr + count > self.neg_table_size:
            self.neg_ptr = 0
            np.random.shuffle(self.neg_table)

        samples = self.neg_table[self.neg_ptr : self.neg_ptr + count]

        self.neg_ptr += count

        return samples
        
    def train_step(self, center_batch, pos_batch, neg_size = 5):
        batch_len = len(center_batch)

        h = self.W1[center_batch]
        v_pos = self.W2[pos_batch]
        neg_idx = self._get_neg_samples(batch_len * neg_size).reshape(batch_len, neg_size)
        v_neg = self.W2[neg_idx]

        pos_scores = np.einsum('ij,ij->i', h, v_pos)
        neg_scores = np.einsum('ij,ikj->ik', h, v_neg)

        pred_pos = self._sigmoid(pos_scores)
        pred_neg = self._sigmoid(neg_scores)

        err_pos = (pred_pos - 1).reshape(-1, 1)
        err_neg = (pred_neg - 0)

        grad_v_pos = err_pos * h
        grad_v_neg = err_neg[:, :, np.newaxis] * h[:, np.newaxis, :]
        grad_h = (err_pos * v_pos) + np.einsum('ik,ikj->ij', err_neg, v_neg)

        np.add.at(self.W1, center_batch, -self.lr * grad_h)
        np.add.at(self.W2, pos_batch, -self.lr * grad_v_pos)
        np.add.at(self.W2, neg_idx.ravel(), -self.lr * grad_v_neg.reshape(-1, self.e_size))

        loss = -np.mean(np.log(pred_pos + 1e-9) + np.sum(np.log(1 - pred_neg + 1e-9), axis=1))

        return loss